# 08 · Failure attribution — every non-exact lift, with evidence

Reads a `lifting_accuracy` run folder (it never runs tblastn). For every scored prediction that is
not base-exact, `valkit.attribution.attribute_cases` records what can be checked on the sequence and
on the record's other predictions, then gives a verdict:

| side | meaning |
|---|---|
| `tool` | the lift is wrong and the sequence shows it |
| `annotation` | the record's own annotation contradicts sequence-level evidence (label swap, merged feature, span that is not a valid CDS) |
| `data` | an N run inside the gene; no lift can be right there |
| `convention` | lift reproduces the reference boundary and the same offset recurs in ≥3 records |
| `undecided` | sequence evidence does not settle it; listed for a human |

Both buckets are attributed: **failed** (cost accuracy) and **coord-only** (counted correct by IoU
or ±6 bp, but not base-exact). The second bucket is where a lenient metric hides mistakes.

Cells marked **[auto]** are generated from the data.

In [ ]:
# Parameters (overridden by validation.valkit.notebooks --param NAME=VALUE)
RUN_ID = "latest"
DATASETS = ["prrs", "fmd", "ped"]


In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for c in [ROOT, *ROOT.parents]:
    if (c / "app" / "src").exists():
        ROOT = c
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from validation.valkit.paths import OUTPUTS

if RUN_ID == "latest":
    runs = sorted(p for p in OUTPUTS.glob("*_lifting_accuracy") if (p / "summary.json").exists())
    RUN_DIR = runs[-1]
else:
    RUN_DIR = OUTPUTS / RUN_ID
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
summary = json.loads((RUN_DIR / "summary.json").read_text())
print("run:", RUN_DIR.name, "| commit", manifest["environment"]["commit"], "|", manifest["environment"]["tblastn"])

frames = []
for ds in DATASETS:
    if ds not in summary["datasets"]:
        continue
    a = pd.read_csv(RUN_DIR / ds / "attribution.tsv", sep="\t")
    a.insert(0, "dataset", ds)
    frames.append(a)
attr = pd.concat(frames, ignore_index=True)
cols = ["dataset", "record_id", "ref_name", "bucket", "side", "category", "pred_start", "pred_end",
        "truth_start", "truth_end", "delta_start", "delta_end", "iou", "max_abs_offset", "offset_recurrence",
        "pred_len_eq_ref", "n_in_uncovered_truth", "truth_cds_valid", "pred_cds_valid",
        "truth_matches_other_pred", "truth_matches_other_iou", "truth_covers_other_preds", "evidence"]
print(len(attr), "non-exact scored predictions")

## Verdicts by bucket, side and category [auto]

In [ ]:
attr.groupby(["dataset", "bucket", "side", "category"]).size().rename("n").reset_index()

## Every failed case

In [ ]:
pd.set_option("display.max_colwidth", 160)
attr[attr["bucket"] == "failed"][cols]

## Coord-only cases that are not conventions

These counted as correct. If `side` is `tool` or `undecided`, the metric was lenient.

In [ ]:
attr[(attr["bucket"] == "coord_only") & (attr["side"] != "convention")][cols]

## Conventions (summarised)

In [ ]:
conv = attr[attr["category"] == "convention_offset"]
conv.groupby(["dataset", "ref_name", "delta_start", "delta_end"]).size().rename("records").reset_index()

## Predictions outside the denominator, checked against the record's full annotation [auto]

In [ ]:
rows = []
for ds in DATASETS:
    if ds not in summary["datasets"]:
        continue
    u = pd.read_csv(RUN_DIR / ds / "unscored_predictions.tsv", sep="\t")
    if len(u):
        u.insert(0, "dataset", ds)
        rows.append(u)
unscored = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if len(unscored):
    display(unscored.groupby(["dataset", "ref_name", "check", "annotated_as"], dropna=False).size().rename("n").reset_index())
    display(unscored[~unscored["check"].isin(["consistent_with_merged", "matches_other_name"])])

## Offset tiers and accuracy by divergence [auto]

In [ ]:
tiers = pd.DataFrame({ds: summary["datasets"][ds]["offset_tiers"] for ds in DATASETS if ds in summary["datasets"]}).T
display(tiers)
ident = pd.concat([pd.DataFrame(summary["datasets"][ds]["accuracy_by_identity"]).assign(dataset=ds)
                   for ds in DATASETS if ds in summary["datasets"]], ignore_index=True)
display(ident)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
pv = attr.pivot_table(index="dataset", columns="side", values="record_id", aggfunc="size", fill_value=0)
pv.plot(kind="barh", stacked=True, ax=ax, color={"tool": "#e76f51", "annotation": "#264653", "data": "#8d99ae",
                                                 "convention": "#e9c46a", "undecided": "#2a9d8f"})
ax.set_xlabel("non-exact scored predictions"); ax.set_title(f"Attribution of non-exact lifts ({RUN_DIR.name})")
fig.tight_layout(); fig.savefig(RUN_DIR / "attribution.png", dpi=160, bbox_inches="tight"); plt.show()

## Summary [auto]

In [ ]:
from IPython.display import Markdown
lines = []
for ds in DATASETS:
    if ds not in summary["datasets"]:
        continue
    a = attr[attr["dataset"] == ds]; s = summary["datasets"][ds]
    f = a[a["bucket"] == "failed"]; c = a[a["bucket"] == "coord_only"]
    lines.append(f"**{s['virus']}** ({s['truth_genes']} scored pairs): {len(f)} failed, {len(c)} coord-only.")
    lines.append(f"- failed by side: {f['side'].value_counts().to_dict()}; by category: {f['category'].value_counts().to_dict()}")
    lines.append(f"- coord-only by side: {c['side'].value_counts().to_dict()}")
    lenient = c[c['side'].isin(['tool', 'undecided'])]
    if len(lenient):
        lines.append(f"- counted correct but not a convention: " + "; ".join(f"{r.record_id} {r.ref_name} ({r.category}, start {r.delta_start:+.0f}, end {r.delta_end:+.0f})" for r in lenient.itertuples()))
    lines.append(f"- offset tiers: {s['offset_tiers']}")
    lines.append(f"- unscored predictions: {s['unscored_predictions']}")
Markdown("\n".join(lines))